In [ ]:
import json
import pandas as pd
import csv
import os
from google import genai
from google.genai import types


def prompt_from_req(o: dict) -> str:
    for root in (o.get("request", o.get("body", o)),):
        if isinstance(root, dict) and isinstance(root.get("contents"), list):
            texts = []
            for c in root["contents"]:
                parts = c.get("parts", []) if isinstance(c, dict) else []
                texts.append("".join(p.get("text", "") for p in parts if isinstance(p, dict)))
            t = "\n".join(x for x in texts if x).strip()
            if t:
                return t
    return o.get("prompt", "") if isinstance(o.get("prompt", ""), str) else ""


def response_from_res(o: dict) -> str:
    try:
        parts = o["response"]["candidates"][0].get("content", {}).get("parts", [])
        return "".join(p.get("text", "") for p in parts if isinstance(p, dict)).strip()
    except Exception:
        return ""


def model_from_res(o: dict) -> str:
    return o.get("response", {}).get("modelVersion", "") or ""

TASK = "anagram"
PROMPTS  = ["multi_step"]

API_KEY = "" # YOUR KEY HERE
MODEL_ID = "gemini-3-pro-preview"

for PROMPT in PROMPTS:
    DATASET_FILE = f"./datasets/{TASK}.csv"
    TEMPLATE_FILE = f"{PROMPT}.txt"
    INPUT_BATCH = f"./input_batch_files/{TASK}/{PROMPT}_batch_requests.jsonl"
    OUTPUT_BATCH = f"./output_batch_files/{TASK}/{PROMPT}_raw_results.jsonl"
    FINAL_RESULT = f"./results/{TASK}/{PROMPT}_results.csv"
    client = genai.Client(api_key=API_KEY)


    with open(f"prompts/{TEMPLATE_FILE}", 'r') as f:
        prompt_prefix = f.read()

    prompt_suffix = "Now, please answer this question and provide your confidence level."
    cot_suffix = " Let's think it step by step."


    print(f"Generating {INPUT_BATCH}...")
    with open(DATASET_FILE, mode='r', encoding='utf-8') as csvfile, \
        open(INPUT_BATCH, 'w', encoding='utf-8') as jsonlfile:
        
        reader = csv.DictReader(csvfile)
        for i, row in enumerate(reader):

            if PROMPT == "self_probing":
                prev_response = row['prev_response']
                formatted_prompt = f"Question: {row['question']}\nPossible Answer: {prev_response}\n{prompt_prefix}"
            elif PROMPT == "vanilla":
                formatted_prompt = f"{prompt_prefix}\nQuestion: {row['question']}\n{prompt_suffix}"
            elif PROMPT == "CoT":
                formatted_prompt = f"{prompt_prefix}\nQuestion: {row['question']}\n{prompt_suffix}{cot_suffix}"
            elif PROMPT == "repetitions" or PROMPT=="formatted" :
                formatted_prompt = f"Question: {row['question']}\n{prompt_prefix}"
            else :
                formatted_prompt = f"{prompt_prefix}\nQuestion: {row['question']}"

            batch_request = {
                "key": row["instance_id"],
                "request": {
                    "contents": [
                        {
                            "role": "user",
                            "parts": [{"text": formatted_prompt}]
                        }
                    ],
                    "generation_config": {
                        "temperature": 0.0,
                        "top_p": 0.0,
                        "top_k": 1,
                    }
                }
            }
            
            jsonlfile.write(json.dumps(batch_request) + '\n')


    print("Uploading JSONL to File API...")
    uploaded_file = client.files.upload(file=INPUT_BATCH,config=types.UploadFileConfig(display_name='my-batch-requests', mime_type='jsonl'))

    print("Creating batch job...")
    batch_job = client.batches.create(
        model=MODEL_ID,
        src=uploaded_file.name,
        config={
            'display_name': "csv-processing-job",
        }
    )

    print(f"Batch job created successfully!")
    print(f"Job Name: {batch_job.name}")
    print(f"Current State: {batch_job.state}")


    import time
    from datetime import datetime



    print(f"Monitoring job: {batch_job.name}")
    print("Note: Batch jobs can take 15 mins to 24 hours.")

    while True:

        batch_job = client.batches.get(name=batch_job.name)

        
        current_state = batch_job.state.name
        timestamp = datetime.now().strftime("%H:%M:%S")
        
        print(f"[{timestamp}] Current Status: {current_state}")

        if current_state == "JOB_STATE_SUCCEEDED":
            print("\n✅ Success! Processing complete.")
            break
        elif current_state in ["JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"]:
            print(f"\n❌ Job ended with status: {current_state}")
            if hasattr(batch_job, 'error'):
                print(f"Error detail: {batch_job.error}")
            break
        
        time.sleep(600)  # 10 minutes

    try:
        result_file = batch_job.dest.file_name
        print(f"Downloading results from {result_file}...")
        
        content = client.files.download(file=result_file)
        with open(OUTPUT_BATCH, "wb") as f:
            f.write(content)
        print("Results saved to raw_results.jsonl")
    except Exception as e:
        print(f"Could not download file: {e}")


    TARGET_DICT = {"addition":"target", "anagram":"word", "locality":"city", "gpqa":"correct_option","openbookqa":"answerKey",
                   "geometry":"answer","logic_puzzles":"answer", "algebra": "answer", "arithmetic":"answer",
                   "combinatory": "answer", "number_theory": "answer"}


    df = pd.read_csv(DATASET_FILE)
    df["key"] = df["instance_id"]
    df["task"] = TASK

    req_rows = []
    with open(INPUT_BATCH, "r", encoding="utf-8") as f:
        for line in f:
            o = json.loads(line)
            req_rows.append({"key": int(o["key"]), "prompt": prompt_from_req(o)})
    df_req = pd.DataFrame(req_rows)

    res_rows = []
    with open(OUTPUT_BATCH, "r", encoding="utf-8") as f:
        for line in f:
            o = json.loads(line)
            res_rows.append({
                "key": int(o["key"]),
                "model": model_from_res(o),
                "response": response_from_res(o),
            })
    df_res = pd.DataFrame(res_rows)

    final = (
        df.merge(df_req, on="key", how="left")
        .merge(df_res, on="key", how="left")
        .loc[:, ["instance_id", "question", "model", "task", TARGET_DICT[TASK], "prompt", "response"]]
    )

    final.to_csv(FINAL_RESULT, sep=";", index=False, encoding="utf-8")
    print("Wrote:", FINAL_RESULT)

